# NYC Collision Severity: Model Training & Batch Transform (Feature Store Integrated)
This notebook demonstrates the training of a Random Forest model using data retrieved directly from the **SageMaker Feature Store** and performing Batch Inferences with input/output joining.

In [ ]:
!pip install "sagemaker<3.0.0" pandas numpy awswrangler

import os
import boto3
import sagemaker
import pandas as pd
import numpy as np
from sagemaker.session import Session
from sagemaker.feature_store.feature_group import FeatureGroup
import awswrangler as wr
import sys

# Setup
role = sagemaker.get_execution_role()
sess = sagemaker.Session()
region = sess.boto_region_name
bucket = sess.default_bucket()
prefix = "aai-540-group6/nyc-collisions-ml"

boto_session = boto3.Session(region_name=region)
sagemaker_client = boto_session.client(service_name="sagemaker", region_name=region)
featurestore_runtime = boto_session.client(
    service_name="sagemaker-featurestore-runtime", region_name=region
)

feature_store_session = Session(
    boto_session=boto_session,
    sagemaker_client=sagemaker_client,
    sagemaker_featurestore_runtime_client=featurestore_runtime,
)

print(f"SageMaker Role: {role}")
print(f"S3 Bucket: {bucket}")

### 1. Retrieve Data from Feature Store
Instead of loading a local CSV, we query the Feature Group we set up in Phase 3. This ensures we are using the most up-to-date engineered features (including the teammate's contributions).

In [ ]:
# Specify the Feature Group Name created in notebook 03
# You may need to update this name to match your active group
feature_group_name = "nyc-collision-group-REPLACE_WITH_DATE_SUFFIX"

collision_feature_group = FeatureGroup(
    name=feature_group_name, sagemaker_session=feature_store_session
)

# Run Athena Query to pull data from Offline Store
query = collision_feature_group.athena_query()
table_name = query.table_name
database_name = query.database

query_string = f'SELECT * FROM "{database_name}"."{table_name}"'
print(f"Running Query: {query_string}")

# Fetch into DataFrame
df = wr.athena.read_sql_query(query_string, database=database_name)
print(f"Retrieved {len(df)} rows from Feature Store.")
display(df.head())

### 2. Data Preparation & Splitting
3-way split: 80% Train, 10% Val, 10% Batch. We use the features retrieved from the Feature Store.

In [ ]:
# Features are already engineered in the Feature Store!
features = ['borough', 'month', 'crash_hour', 'rush_hour', 'is_weekend', 'factor_category', 'vehicle_type']
target = 'target'
identifier = 'collision_id'

# Perform split
rand_split = np.random.rand(len(df))
train_idx = rand_split < 0.8
val_idx = (rand_split >= 0.8) & (rand_split < 0.9)
batch_idx = rand_split >= 0.9

data_train = df[train_idx][features + [target]]
data_val = df[val_idx][features + [target]]
data_batch = df[batch_idx][features + [identifier]] # Keep ID for joining

print(f"Train: {len(data_train)}, Val: {len(data_val)}, Batch: {len(data_batch)}")

### 3. Upload Data to S3

In [ ]:
os.makedirs('tmp', exist_ok=True)

data_train.to_csv('tmp/train.csv', index=False, header=False)
data_val.to_csv('tmp/validation.csv', index=False, header=False)
data_batch.to_csv('tmp/batch.csv', index=False, header=False)

train_s3 = sess.upload_data('tmp/train.csv', bucket, f"{prefix}/train")
val_s3 = sess.upload_data('tmp/validation.csv', bucket, f"{prefix}/validation")
batch_s3 = sess.upload_data('tmp/batch.csv', bucket, f"{prefix}/batch")

print(f"Train S3: {train_s3}")

### 4. SageMaker Training Job
Using our `sagemaker_train.py` script.

In [ ]:
from sagemaker.sklearn.estimator import SKLearn

sklearn_estimator = SKLearn(
    entry_point='sagemaker_train.py',
    source_dir='../src',
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge', 
    framework_version='1.2-1',
    py_version='py3',
    hyperparameters={
        'n-estimators': 100,
        'max-depth': 10
    }
)

sklearn_estimator.fit({'train': train_s3})
print("Training complete.")

### 5. Batch Transform with Input/Output Joining
We associate predictions with `collision_id` using Assignment 4.1 techniques.

In [ ]:
transformer = sklearn_estimator.transformer(
    instance_count=1,
    instance_type='ml.m5.xlarge',
    output_path=f"s3://{bucket}/{prefix}/batch_output",
    accept='text/csv',
    assemble_with='Line'
)

transformer.transform(
    data=batch_s3,
    content_type='text/csv',
    split_type='Line',
    input_filter='$[1:]', # Send only features to model (exclude ID at index 0)
    join_source='Input', 
    output_filter='$[0,-1]' # Output ID and Prediction
)

print("Batch Transform job started.")
transformer.wait()

### 6. Final Results from S3
Download and display the results from the Datalake.

In [ ]:
output_key = f"{prefix}/batch_output/batch.csv.out"
results = wr.s3.read_csv(f"s3://{bucket}/{output_key}", header=None)
results.columns = ['collision_id', 'predicted_severity']

print("--- Final Batch Predictions with Identifiers ---")
display(results.head(10))